## Unified Document Loader

**What it does:** A single function that detects the file type and uses the right parser automatically. 

**When to use it:** In production, when you receive files of different types and want one function to handle them all. 

**Steps:** 
1. Check the file extension (`.pdf`, `.html`, `.md`, `.txt`).
2. Call the matching parser.
3. Return a list of `{text, metadata}` dictionaries. 

In [8]:
import os
import fitz
from bs4 import BeautifulSoup
import json

In [13]:
pdf_path = "./tmp/VyshaliSamplePDF.pdf"
sample_md = "./tmp/sample_nb.md"

In [ ]:
def load_document(file_path):

    file_name = os.path.basename(file_path)
    
    print(file_name)

    print(os.path.splitext(file_path))

    file_extension = os.path.splitext(file_path)[1]
    file_extension = file_extension.lower()


    if file_extension == ".pdf":
        doc = fitz.open(file_path)

        pages = []

        for i in range(len(doc)):
            page_text = doc[i].get_text()
            page_text = page_text.strip()

            if page_text:
                metadata = {
                    "src": file_name,
                    "page": i+1
                }
                page_info = {
                    "text": page_text,
                    "meta": metadata
                }
                pages.append(page_info)
        
        doc.close()
        return pages
    

    elif file_extension in (".html", ".htm"):
        with open(file_path) as f:
            html_content = f.read()
            print(html_content)

        soup = BeautifulSoup(html_content, "html.parser")

        for tag in soup(["script", "style", "nav", "footer"]):
            tag.decompose()

        clean_text = soup.get_text(separator="\n", strip=True)

        metadata = {
            "src": file_name,
            "type": "html"
        }

        return [{
            "text": clean_text,
            "meta": metadata
        }]
    

    elif file_extension == ".md":
        with open(file_path) as f:
            content = f.read()

        metadata = {
            "src": file_name,
            "type": "md"
        }

        return [{
            "text": content,
            "meta": metadata
        }]
    

    else:
        with open(file_path) as f:
            content = f.read()

        metadata = {
            "src": file_name,
            "type": file_extension
        }

        return [{
            "text": content,
            "meta": metadata
        }]



load_document(pdf_path)

VyshaliSamplePDF.pdf
('./tmp/VyshaliSamplePDF', '.pdf')


[{'text': 'RAG Document Guide\nChapter 1: Introduction\nRAG combines retrieval with generation.\nChapter 2: Steps\n1. Parse documents  2. Chunk  3. Embed  4. Retrieve',
  'meta': {'src': 'VyshaliSamplePDF.pdf', 'page': 1}}]

**What happened:** The function checked the file extension, picked the PDF parser, and returned structured output.

In [ ]:
os.remove(pdf_path)
os.remove(sample_md)

## Summary

- **PyMuPDF** reads PDF files. Use `get_text()` to extract text from each page.
- **BeautifulSoup** reads HTML. Use `decompose()` to remove unwanted tags first. 
- HTML tables can be converted to lists of dictionaries using `find_all("tr")`.
- Markdown files can be split into sections by splitting on heading lines.
- A **unified document loader** detects file type and picks the right parser.
- Always return a consistent format: a list of `{text, metadata}` dictionaries. 